In [ ]:
import pandas as pd
import csv
import numpy as np
import os
from textblob import TextBlob
import nltk
import re
import string
from nltk.corpus import stopwords

In [ ]:
try:
    df = pd.read_csv("final_data.csv",
                    quoting=csv.QUOTE_MINIMAL,
                    on_bad_lines='warn',
                    engine='python')
    print("Dosya başarıyla yüklendi")
except Exception as e:
    df = pd.read_csv("final_data.csv",
                    on_bad_lines='skip',
                    engine='c')
    print("Bazı hatalı satırlar atlanarak yüklendi")

/tmp/ipython-input-1674388352.py:2: ParserWarning: Skipping line 124: field larger than field limit (131072)

  df = pd.read_csv("final_data.csv",
/tmp/ipython-input-1674388352.py:2: ParserWarning: Skipping line 2595: field larger than field limit (131072)

  df = pd.read_csv("final_data.csv",
/tmp/ipython-input-1674388352.py:2: ParserWarning: Skipping line 3008: field larger than field limit (131072)

  df = pd.read_csv("final_data.csv",
/tmp/ipython-input-1674388352.py:2: ParserWarning: Skipping line 7758: field larger than field limit (131072)

  df = pd.read_csv("final_data.csv",
/tmp/ipython-input-1674388352.py:2: ParserWarning: Skipping line 7794: ',' expected after '"'

  df = pd.read_csv("final_data.csv",
/tmp/ipython-input-1674388352.py:2: ParserWarning: Skipping line 7884: ',' expected after '"'

  df = pd.read_csv("final_data.csv",
/tmp/ipython-input-1674388352.py:2: ParserWarning: Skipping line 2846: Expected 16 fields in line 2846, saw 19

  df = pd.read_csv("final_data.cs

Dosya başarıyla yüklendi


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12354 entries, 0 to 12353
Data columns (total 16 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   id            11251 non-null  object
 1   news_url      12301 non-null  object
 2   title         12282 non-null  object
 3   tweet_ids     10424 non-null  object
 4   label         12257 non-null  object
 5   clean_title   11139 non-null  object
 6   full_text     12235 non-null  object
 7   source        1111 non-null   object
 8   word_count    12228 non-null  object
 9   sentiment     12227 non-null  object
 10  subjectivity  12225 non-null  object
 11  lexical_div   12225 non-null  object
 12  caps_ratio    12225 non-null  object
 13  excl_count    12224 non-null  object
 14  label_num     12224 non-null  object
 15  label_str     12224 non-null  object
dtypes: object(16)
memory usage: 1.5+ MB


In [ ]:
print(df.columns)
display(df.head(5))

Index(['id', 'news_url', 'title', 'tweet_ids', 'label', 'clean_title',
       'full_text', 'source', 'word_count', 'sentiment', 'subjectivity',
       'lexical_div', 'caps_ratio', 'excl_count', 'label_num', 'label_str'],
      dtype='object')


,id,news_url,title,tweet_ids,label,clean_title,full_text,source,word_count,sentiment,subjectivity,lexical_div,caps_ratio,excl_count,label_num,label_str
0,gossipcop-897560,https://www.elitedaily.com/p/the-2018-holiday-...,"The 2018 Holiday Gift For Your Partner, Based ...",936685609279737863\t936686125288185856\t936686...,0,2018 holiday gift partner based zodiac sign,"Since we're in the midst of Hanukkah, and Chri...",NaN,859.0,0.19346397649969077,0.513658266336838,0.5448195576251456,0.02273172643543014,1.0,0,0
1,gossipcop-849067,https://www.refinery29.com/en-us/2017/05/15331...,Gen-Z Takes The MTV Movie & TV Awards Red Carp...,861370421601595392\t861370843342831616\t861370...,0,genz takes mtv movie awards red carpet storm,Entertainment\n\nHow Michaela Coel Pulled Off ...,NaN,37.0,0.0,0.0,1.0,0.08620689655172414,0.0,0,0
2,gossipcop-875696,https://en.wikipedia.org/wiki/Look_What_You_Ma...,Look What You Made Me Do,901100538560991232\t901968588210393088\t901969...,0,look made,"2017 single by Taylor Swift\n\n""Look What You ...",NaN,5396.0,0.09522159556250459,0.4031966211511663,0.37268346923647144,0.03683263713144495,4.0,0,0
3,gossipcop-897997,https://www.yahoo.com/lifestyle/selena-gomez-w...,Selena Gomez Wore 6 Outfits in 1 Day — But She...,937821202885648389,0,selena gomez wore outfits day loved date night...,Selena Gomez and the Weeknd stepped out for da...,NaN,320.0,0.12593795093795093,0.39029581529581525,0.621875,0.04902477596204533,0.0,0,0
4,gossipcop-844614,https://www.biography.com/people/eric-roberts-...,Eric Roberts,854785357421957120,0,eric roberts,(1956-)\n\nSynopsis\n\nActor Eric Roberts was ...,NaN,747.0,0.018086935895155067,0.3466346438949179,0.5194109772423026,0.0497545738509594,1.0,0,0


In [ ]:

try:
    welfake = pd.read_csv("WELFake_Dataset.csv",
                         engine='python',
                         on_bad_lines='skip',
                         quoting=csv.QUOTE_MINIMAL)
    print(f"Yükleme tamamlandı.Toplam satır: {len(welfake)}")
except Exception as e:
    print(f"Hata: {e}")

Yükleme tamamlandı.Toplam satır: 72154


In [ ]:
# 2. VERİ TEMİZLİĞİ VE FİLTRELEME
# Label sütununu kontrol ediyoruz ve boşları siliyoruz
welfake = welfake.dropna(subset=['label', 'text'])
welfake['label'] = pd.to_numeric(welfake['label'], errors='coerce')
welfake = welfake.dropna(subset=['label'])
welfake['label'] = welfake['label'].astype(int)

# sahte haberler için
new_fakes = welfake[welfake['label'] == 0].copy()
new_fakes['label'] = 1 # fake haberler 1

available_fakes = len(new_fakes)
print(f"🔍 Sistemde bulunan gerçek toplam Fake haber sayısı: {available_fakes}")

# verisetinden yalan haberleri alıyoruz
if available_fakes >= 9000:
    new_fakes = new_fakes.sample(n=9000, random_state=42)
    print(f" 9000 tane sahte haber alındı.")
else:
    print(f"Sadece {available_fakes} adet fake haber bulundu, bulunanların tamamı alınıyor.")

new_fakes = new_fakes.rename(columns={'text': 'full_text'})  # orjinal final veri seti ile aynı olması için sütunları eşitliyoruz

🔍 Sistemde bulunan gerçek toplam Fake haber sayısı: 35028
 9000 tane sahte haber alındı.


In [ ]:
def analyze_text_style(text):  # analiz fonksiyonu
    text = str(text)
    if not isinstance(text, str) or len(text) < 5:
        return pd.Series([0, 0, 0, 0, 0, 0])

    words = text.split()
    num_words = len(words) if len(words) > 0 else 1

    # Sentiment & Subjectivity
    try:
        blob = TextBlob(text)
        sentiment = blob.sentiment.polarity
        subjectivity = blob.sentiment.subjectivity
    except:
        sentiment, subjectivity = 0, 0

    # Lexical Diversity
    unique_words = len(set(words))
    lexical_div = unique_words / num_words

    # Uppercase & Punctuation
    caps_count = sum(1 for c in text if c.isupper())
    caps_ratio = caps_count / len(text) if len(text) > 0 else 0
    exclamation_count = text.count('!')

    return pd.Series([num_words, sentiment, subjectivity, lexical_div, caps_ratio, exclamation_count])

# kendi veri setimizdeki kolonları düzenliyoruz
numeric_cols = ['label', 'word_count', 'sentiment', 'subjectivity', 'lexical_div', 'caps_ratio', 'excl_count']
for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

for col in df.columns:
    if col not in new_fakes.columns: # ve yeni aldığımız sahte haberlerle birleştiriyoruz
        new_fakes[col] = np.nan

df_final = pd.concat([df, new_fakes[df.columns]], ignore_index=True)  # final veri eski verisetimiz ile yeni verilerin birleşmiş hali oluyor

# kopya var mı diye temizlik yapıyoruz
df_final.drop_duplicates(subset=['full_text'], inplace=True)
df_final = df_final.dropna(subset=['label', 'full_text'])
df_final['label'] = df_final['label'].astype(int)

print(f"Yeni Veri Seti Durumu:")
print(f"Toplam Satır: {len(df_final)}")  # oluşan final veristeinin satırları
print(f"Sahte Haber (Label 1): {len(df_final[df_final['label']==1])}")  # yalan haber sayısı
print(f"Gerçek Haber (Label 0): {len(df_final[df_final['label']==0])}") # gerçek haber sayısı

# eksik özellikleri kontrol ediyoruz
print("Yeni verisetinin stylometric özellikleri hesaplanıyor..")
mask = df_final['sentiment'].isna()
df_final.loc[mask, ['word_count', 'sentiment', 'subjectivity', 'lexical_div', 'caps_ratio', 'excl_count']] = \
    df_final.loc[mask, 'full_text'].apply(analyze_text_style)

df_final.to_csv("final_data2.csv", index=False) # yeni csv dosyası olarak kaydediyoruz
print("Veriseti kaydedildi.")

df = df_final

Yeni Veri Seti Durumu:
Toplam Satır: 20405
Sahte Haber (Label 1): 9949
Gerçek Haber (Label 0): 10456
⏳ Yeni verisetinin stylometric özellikleri hesaplanıyor..
Veriseti kaydedildi.


In [ ]:
df_final.head(10)

,id,news_url,title,tweet_ids,label,clean_title,full_text,source,word_count,sentiment,subjectivity,lexical_div,caps_ratio,excl_count,label_num,label_str
0,gossipcop-897560,https://www.elitedaily.com/p/the-2018-holiday-...,"The 2018 Holiday Gift For Your Partner, Based ...",936685609279737863\t936686125288185856\t936686...,0,2018 holiday gift partner based zodiac sign,"Since we're in the midst of Hanukkah, and Chri...",NaN,859.0,0.193464,0.513658,0.544820,0.022732,1.0,0,0
1,gossipcop-849067,https://www.refinery29.com/en-us/2017/05/15331...,Gen-Z Takes The MTV Movie & TV Awards Red Carp...,861370421601595392\t861370843342831616\t861370...,0,genz takes mtv movie awards red carpet storm,Entertainment\n\nHow Michaela Coel Pulled Off ...,NaN,37.0,0.000000,0.000000,1.000000,0.086207,0.0,0,0
2,gossipcop-875696,https://en.wikipedia.org/wiki/Look_What_You_Ma...,Look What You Made Me Do,901100538560991232\t901968588210393088\t901969...,0,look made,"2017 single by Taylor Swift\n\n""Look What You ...",NaN,5396.0,0.095222,0.403197,0.372683,0.036833,4.0,0,0
3,gossipcop-897997,https://www.yahoo.com/lifestyle/selena-gomez-w...,Selena Gomez Wore 6 Outfits in 1 Day — But She...,937821202885648389,0,selena gomez wore outfits day loved date night...,Selena Gomez and the Weeknd stepped out for da...,NaN,320.0,0.125938,0.390296,0.621875,0.049025,0.0,0,0
4,gossipcop-844614,https://www.biography.com/people/eric-roberts-...,Eric Roberts,854785357421957120,0,eric roberts,(1956-)\n\nSynopsis\n\nActor Eric Roberts was ...,NaN,747.0,0.018087,0.346635,0.519411,0.049755,1.0,0,0
5,gossipcop-892932,https://en.wikipedia.org/wiki/Paul_Anka,Jason Bateman Defends Wife Amanda Anka After S...,928879688545656832\t928880037142646784\t928880...,0,jason bateman defends wife amanda anka accuses...,Canadian-American singer and actor (born 1941)...,NaN,2567.0,0.122129,0.345941,0.460849,0.056263,3.0,0,0
6,gossipcop-891312,https://variety.com/2018/tv/news/robin-wright-...,Robin Wright on Kevin Spacey: ‘House of Cards’...,926417709457436672\t926417975086862337\t926418...,0,robin wright kevin spacey ‘house cards’ team ‘...,“House of Cards” star Robin Wright spoke out M...,NaN,191.0,0.028571,0.359524,0.717277,0.031667,0.0,0,0
7,politifact15158,https://web.archive.org/web/20180327213804/htt...,BREAKING: Irish superstar Saoirse Ronan dies a...,NaN,1,breaking irish superstar saoirse ronan dies on...,Irish and American actress Saoirse Ronan actre...,NaN,455.0,0.041349,0.444841,0.569231,0.041652,0.0,1,1
8,NaN,https://www.thebeaverton.com/2025/02/tampax-en...,Tampax encourages women to channel their rage ...,NaN,1,NaN,LOS ANGELES – With Donald Trump’s second presi...,The Beaverton,345.0,0.189572,0.459358,0.695652,0.024531,0.0,1,1
9,gossipcop-858874,https://www.usatoday.com/story/life/music/2017...,"After London terror attack, Ariana Grande conc...",871374813322989568\t871375034303959040\t871375...,0,london terror attack ariana grande concert man...,Hours after a second terror attack claimed the...,NaN,353.0,0.139903,0.370846,0.651558,0.043972,1.0,0,0


In [ ]:
#bu işlemlerden sonra bazı sütunlarda eksik veri oluyor, onları temizlememiz gerekiyor
print("Eksik veri içeren sütunlar:")
print(df.isnull().sum())

# 2boş olan hesaplanmamış satırları tekrar hesaplıyoruz
def fill_missing(row):
    if pd.isna(row['sentiment']):
        return analyze_text_style(row['full_text'])
    else:
        return pd.Series([row['word_count'], row['sentiment'], row['subjectivity'], row['lexical_div'], row['caps_ratio'], row['excl_count']])

df[['word_count', 'sentiment', 'subjectivity', 'lexical_div', 'caps_ratio', 'excl_count']] = \
    df.apply(fill_missing, axis=1)

print("İşlem tamamlandı ve eksik sütunların verileri dolduruldu")

Eksik veri içeren sütunlar:
id               9827
news_url         8936
title               0
tweet_ids       10525
label               0
clean_title      9827
full_text           0
source          19510
word_count       8934
sentiment        8934
subjectivity     8936
lexical_div      8936
caps_ratio       8936
excl_count       8936
label_num        8936
label_str        8936
dtype: int64
İşlem tamamlandı ve eksik sütunların verileri dolduruldu


In [ ]:
from scipy import stats
import numpy as np

# 1. Analiz edilecek sütunlardaki NaN ve Inf değerleri temizleyelim
features = ['sentiment', 'subjectivity', 'lexical_div', 'word_count']

# Sadece bu sütunları içeren ve boş olmayan bir kopya oluşturalım
df_clean = df.dropna(subset=features)

print(f"Analiz edilen temiz satır sayısı: {len(df_clean)}")

# 2. Testi tekrar çalıştıralım
for f in features:
    fake = df_clean[df_clean['label'] == 1][f]
    real = df_clean[df_clean['label'] == 0][f]

    # Grupların boş olmadığını kontrol et
    if len(fake) > 0 and len(real) > 0:
        t_stat, p_val = stats.ttest_ind(fake, real, nan_policy='omit')
        print(f"{f:15} | P-Value: {p_val:.10f} | {'SIGNIFICANT' if p_val < 0.05 else 'NOT SIGNIFICANT'}")
    else:
        print(f"{f:15} | Hata: Gruplardan biri boş!")

Analiz edilen temiz satır sayısı: 20403
sentiment       | P-Value: 0.0000000000 | SIGNIFICANT
subjectivity    | P-Value: 0.0000000000 | SIGNIFICANT
lexical_div     | P-Value: 0.1226909667 | NOT SIGNIFICANT
word_count      | P-Value: 0.0000000000 | SIGNIFICANT


Lexical diversity nin yüksek pvalue değerine sahip olması yalan haberlerin de profesyonelce hazırlanıp fazla kelime çeşitliliğine sahip olabilidğini gösteriyor.Yani anlamlı bir fark bulunamıyor

Ayrıca sentiment ,subjectivity ve word count pvalue değerlerinin sıfır olması,gerçek ve sahte haberlerin istatistiksel olarak anlamlı olduğunu gösteriyor.

In [ ]:
print("Final verisetinin haber doğruluk dağılımı")
print(df_clean['label'].value_counts())

Final verisetinin haber doğruluk dağılımı
label
0    10454
1     9949
Name: count, dtype: int64


Sonuç olarak yalan haber sayımızı gerçek haberlerle neredeyse aynı yaptık ki dengeli bir model oluşturabilelim.

In [ ]:
df_final

,id,news_url,title,tweet_ids,label,clean_title,full_text,source,word_count,sentiment,subjectivity,lexical_div,caps_ratio,excl_count,label_num,label_str
0,gossipcop-897560,https://www.elitedaily.com/p/the-2018-holiday-...,"The 2018 Holiday Gift For Your Partner, Based ...",936685609279737863\t936686125288185856\t936686...,0,2018 holiday gift partner based zodiac sign,"Since we're in the midst of Hanukkah, and Chri...",NaN,859.0,0.193464,0.513658,0.544820,0.022732,1.0,0,0
1,gossipcop-849067,https://www.refinery29.com/en-us/2017/05/15331...,Gen-Z Takes The MTV Movie & TV Awards Red Carp...,861370421601595392\t861370843342831616\t861370...,0,genz takes mtv movie awards red carpet storm,Entertainment\n\nHow Michaela Coel Pulled Off ...,NaN,37.0,0.000000,0.000000,1.000000,0.086207,0.0,0,0
2,gossipcop-875696,https://en.wikipedia.org/wiki/Look_What_You_Ma...,Look What You Made Me Do,901100538560991232\t901968588210393088\t901969...,0,look made,"2017 single by Taylor Swift\n\n""Look What You ...",NaN,5396.0,0.095222,0.403197,0.372683,0.036833,4.0,0,0
3,gossipcop-897997,https://www.yahoo.com/lifestyle/selena-gomez-w...,Selena Gomez Wore 6 Outfits in 1 Day — But She...,937821202885648389,0,selena gomez wore outfits day loved date night...,Selena Gomez and the Weeknd stepped out for da...,NaN,320.0,0.125938,0.390296,0.621875,0.049025,0.0,0,0
4,gossipcop-844614,https://www.biography.com/people/eric-roberts-...,Eric Roberts,854785357421957120,0,eric roberts,(1956-)\n\nSynopsis\n\nActor Eric Roberts was ...,NaN,747.0,0.018087,0.346635,0.519411,0.049755,1.0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21348,NaN,NaN,Ashton Kutcher Blows Kiss to John McCain Durin...,NaN,1,NaN,Actor Ashton Kutcher sent a little love to Sen...,NaN,444.0,0.056250,0.457143,0.621622,0.028169,0.0,NaN,NaN
21349,NaN,NaN,"Islamic State's Baghdadi, in undated audio, ur...",NaN,1,NaN,CAIRO (Reuters) - Islamic State leader Abu Bak...,NaN,560.0,0.017294,0.268190,0.573214,0.034994,0.0,NaN,NaN
21350,NaN,NaN,Senators say effort to protect 'Dreamers' maki...,NaN,1,NaN,WASHINGTON (Reuters) - A bipartisan push in th...,NaN,639.0,0.034516,0.432459,0.575900,0.032997,0.0,NaN,NaN
21351,NaN,NaN,Nordic states step up defense cooperation beca...,NaN,1,NaN,HELSINKI (Reuters) - Nordic countries agreed o...,NaN,338.0,0.013095,0.314105,0.618343,0.044536,0.0,NaN,NaN


WELFake verisetinden aldığımız haberlerde label_num ve label_str sütunları olmadığı için hepsi null.Ama haber labellarımız doğru şuan.Ayrıca o kolonlar gereksiz olduğu için sonraki aşamada kaldırmamız iyi olacak.

In [ ]:
cols_to_drop = ['label_num', 'label_str', 'id', 'tweet_ids', 'source', 'news_url']
df_final = df_final.drop(columns=[c for c in cols_to_drop if c in df_final.columns])

Ayrıca artık modeli sınıflandırma ve eğitme kısmımızda işe yaramayacak id,tweet id ,source ve url gibi diğer kolonları da kaldırdık.

Şimdi yeni eklediğimiz haberler için clean title kolonlarını doldurmalıyız

In [ ]:
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [ ]:
def clean_title_logic(text):  # önceki temizleme fonksiyonumuz
    if pd.isna(text) or text == "":
        return ""

    text = str(text).lower()  # büyük harfleri küçültür
    text = text.translate(str.maketrans('', '', string.punctuation))  # noktalamaları kaldırır
    text = re.sub(r'\d+', '', text) # boşluk ve sayıları da kaldırır

    words = text.split()
    cleaned_words = [w for w in words if w not in stop_words and len(w) > 2] # 3 karakterden kısa olan kelimeleri atıyoruz

    return " ".join(cleaned_words)

In [ ]:
mask = df_final['clean_title'].isna()  # sadece NaN olanları dolduracağımız için
df_final.loc[mask, 'clean_title'] = df_final.loc[mask, 'title'].apply(clean_title_logic)

In [ ]:
print(f"Sonuç olarak boş kalan clean title kolon sayısı: {df_final['clean_title'].isna().sum()}")


Sonuç olarak boş kalan clean title kolon sayısı: 0


In [ ]:
print("Örnek Temizlenmiş Başlıklar:")
display(df_final[['title', 'clean_title']].tail(5))

Örnek Temizlenmiş Başlıklar:


,title,clean_title
21348,Ashton Kutcher Blows Kiss to John McCain Durin...,ashton kutcher blows kiss john mccain senate h...
21349,"Islamic State's Baghdadi, in undated audio, ur...",islamic states baghdadi undated audio urges mi...
21350,Senators say effort to protect 'Dreamers' maki...,senators say effort protect dreamers making pr...
21351,Nordic states step up defense cooperation beca...,nordic states step defense cooperation russia ...
21353,"Trump, Ryan tout unity in wake of meeting",trump ryan tout unity wake meeting


In [ ]:
df_final.to_csv("final_data2.csv", index=False)


In [ ]:
df_final

,title,label,clean_title,full_text,word_count,sentiment,subjectivity,lexical_div,caps_ratio,excl_count
0,"The 2018 Holiday Gift For Your Partner, Based ...",0,2018 holiday gift partner based zodiac sign,"Since we're in the midst of Hanukkah, and Chri...",859.0,0.193464,0.513658,0.544820,0.022732,1.0
1,Gen-Z Takes The MTV Movie & TV Awards Red Carp...,0,genz takes mtv movie awards red carpet storm,Entertainment\n\nHow Michaela Coel Pulled Off ...,37.0,0.000000,0.000000,1.000000,0.086207,0.0
2,Look What You Made Me Do,0,look made,"2017 single by Taylor Swift\n\n""Look What You ...",5396.0,0.095222,0.403197,0.372683,0.036833,4.0
3,Selena Gomez Wore 6 Outfits in 1 Day — But She...,0,selena gomez wore outfits day loved date night...,Selena Gomez and the Weeknd stepped out for da...,320.0,0.125938,0.390296,0.621875,0.049025,0.0
4,Eric Roberts,0,eric roberts,(1956-)\n\nSynopsis\n\nActor Eric Roberts was ...,747.0,0.018087,0.346635,0.519411,0.049755,1.0
...,...,...,...,...,...,...,...,...,...,...
21348,Ashton Kutcher Blows Kiss to John McCain Durin...,1,ashton kutcher blows kiss john mccain senate h...,Actor Ashton Kutcher sent a little love to Sen...,444.0,0.056250,0.457143,0.621622,0.028169,0.0
21349,"Islamic State's Baghdadi, in undated audio, ur...",1,islamic states baghdadi undated audio urges mi...,CAIRO (Reuters) - Islamic State leader Abu Bak...,560.0,0.017294,0.268190,0.573214,0.034994,0.0
21350,Senators say effort to protect 'Dreamers' maki...,1,senators say effort protect dreamers making pr...,WASHINGTON (Reuters) - A bipartisan push in th...,639.0,0.034516,0.432459,0.575900,0.032997,0.0
21351,Nordic states step up defense cooperation beca...,1,nordic states step defense cooperation russia ...,HELSINKI (Reuters) - Nordic countries agreed o...,338.0,0.013095,0.314105,0.618343,0.044536,0.0


In [ ]:
df_final.info()

<class 'pandas.core.frame.DataFrame'>
Index: 20405 entries, 0 to 21353
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   title         20405 non-null  object 
 1   label         20405 non-null  int64  
 2   clean_title   20405 non-null  object 
 3   full_text     20405 non-null  object 
 4   word_count    20405 non-null  float64
 5   sentiment     20405 non-null  float64
 6   subjectivity  20403 non-null  float64
 7   lexical_div   20403 non-null  float64
 8   caps_ratio    20403 non-null  float64
 9   excl_count    20403 non-null  float64
dtypes: float64(6), int64(1), object(3)
memory usage: 1.7+ MB


GENEL OLARAK NOTEBOOK ÖZETİ:

Veri Seti Birleştirme: Mevcut haber veri seti olan WELFake veriseti yüklenir ve yapısal olarak incelenir.

Sınıf Dengelenmesi: Başlangıçta daha az olan "sahte haber" (label 1) sınıfını dengelemek için çekilen bu yeni veriler ana veri setine eklenir.

Temizlik ve Ön İşleme: Metinler temizlenir, küçük harfe çevrilir ve temel kelime sayıları gibi ilk özellikler hesaplanır.

Sonuç: Analize hazır, dengeli ve temizlenmiş df_final veri seti oluşturulur.